# Multi-Agent Product Pricer

This notebook implements a simple **multi-agent AI pricing system** using only open-source tools.

Architecture:

User Input  
↓  
Scanner Agent (ChromaDB + embeddings)  
↓  
Ollama Agent 1 (Llama3)  
Ollama Agent 2 (Mistral)  
↓  
Planner Agent (ensemble pricing)  
↓  
Gradio UI

To make the system reliable, the LLMs return **structured JSON responses**:

{"price": number}

In [11]:
import re
import json
import requests
import chromadb
import gradio as gr
from sentence_transformers import SentenceTransformer

OLLAMA_URL = "http://localhost:11434/api/generate"

print("Environment ready.")

Environment ready.


In [12]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.get_or_create_collection("products")

products = [
    "Apple MacBook Pro M2 16GB RAM 512GB SSD",
    "Samsung 65 inch 4K Smart TV",
    "Sony WH-1000XM5 Noise Cancelling Headphones",
    "Dell XPS 13 Laptop 16GB RAM",
    "Apple iPhone 14 Pro 256GB",
]

embeddings = embedder.encode(products)

collection.add(
    documents=products,
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in range(len(products))]
)

print("Vector database ready.")

Vector database ready.


In [13]:
def parse_price(text):
    try:
        data = json.loads(text)
        return float(data.get("price", 0))
    except:
        numbers = re.findall(r"\d+(?:\.\d+)?", text)
        if numbers:
            return float(numbers[0])
        return 0.0


def ollama_price(model, description):

    prompt = f"""
You are a pricing engine.

Estimate the market price of the product below.

Return ONLY valid JSON in this format:

{{"price": number}}

Example:
{{"price": 1200}}

Product:
{description}
"""

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    try:
        r = requests.post(OLLAMA_URL, json=payload)
        text = r.json().get("response","")

        print(f"\nModel: {model}")
        print("Response:", text)

        return parse_price(text)

    except Exception as e:
        print("Error:", e)
        return 0.0

In [14]:
def scanner_agent(description):

    query_embedding = embedder.encode([description])

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3
    )

    return results["documents"][0]


def ensemble_price(description):

    similar_products = scanner_agent(description)

    price_llama = ollama_price("llama3", description)
    price_mistral = ollama_price("mistral", description)

    price_llama = float(price_llama)
    price_mistral = float(price_mistral)

    final_price = (0.6 * price_llama) + (0.4 * price_mistral)

    return {
        "similar_products": similar_products,
        "llama3": price_llama,
        "mistral": price_mistral,
        "final": round(final_price,2)
    }

In [ ]:
def run(description):

    if not description:
        return "",0,0,0

    result = ensemble_price(description)

    similar = "\n".join(result["similar_products"])

    return (
        similar,
        result["llama3"],
        result["mistral"],
        result["final"]
    )


with gr.Blocks() as app:

    gr.Markdown("# Open Source AI Price Estimator")

    inp = gr.Textbox(label="Product Description")

    similar = gr.Textbox(label="Similar Products (RAG)")
    llama = gr.Number(label="Llama3 Estimate")
    mistral = gr.Number(label="Mistral Estimate")
    final = gr.Number(label="Final Estimate")

    btn = gr.Button("Estimate")

    btn.click(
        run,
        inputs=inp,
        outputs=[similar, llama, mistral, final]
    )

app.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.



Model: llama3
Response: 

Model: mistral
Response: 

Model: llama3
Response: 

Model: mistral
Response: 
